In [1]:
import json
import tqdm
import pandas as pd
from ollama import chat
from pydantic import BaseModel, ValidationError
from typing import List, Optional

class Product(BaseModel):
    anklage: bool
    #text: str



In [2]:
def read_jsonl(file_path):
    """Read a jsonl file and return a list of records."""
    print(f'\n\n#### Reading "{file_path}" for training / test data (80/20 split) ####\n\n')
    records = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
                
    return records

In [3]:
def load_data(input_path):
    # Loading data, renaming columns to what trainer expects, and converting to Dataset
    data_records = read_jsonl(input_path)
    data = pd.DataFrame(data_records)
    data = data[["text"]]

    return data

In [4]:
orig_data = load_data('C:/dev/bachelor/Bachelor_project/data/training_data/validation_set/validation_set.jsonl') 



#### Reading "C:/dev/bachelor/Bachelor_project/data/training_data/validation_set/validation_set.jsonl" for training / test data (80/20 split) ####




In [7]:
test_data = orig_data["text"][2:3]

In [ ]:

results = []

for idx, sentence in enumerate(tqdm.tqdm(orig_data, desc="Blame detection")):
    retries = 2
    parsed = None
    
    for attempt in range(retries):
        try:
            response = chat(
                model='qwen3.5:9b',
                messages=[
                    {
                        'role': 'system',
                        'content': 'Du er en ekspert i at identificere hvornår politikere anklager hinanden for at være skyld i et negativt udfald. Identificér om der er nogle der anklager hinanden i sætningen. /no_think'
                    },
                    {
                        'role': 'user',
                        'content': f'{sentence}'
                    }
                ],
                format=Product.model_json_schema(),
                options={'temperature': 0}
            )
            
            raw = response.message.content
            
            if not raw or not raw.strip():
                raise ValueError(f"Empty response on attempt {attempt + 1}")
            
            parsed = Product.model_validate_json(raw)
            break
            
        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            print(f"\n[Attempt {attempt + 1}] Parse error for sentence: '{sentence[:60]}...'\n  Error: {e}")
            if attempt == retries - 1:
                print(f"  → Skipping after {retries} failed attempts.")
                parsed = None
    
    results.append({
        'idx': idx,
        'sentence': sentence,
        'anklage': parsed.anklage if parsed else None
    })

with open('blame_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} results to blame_results.json")

Blame detection:   0%|          | 0/424 [00:00<?, ?it/s]